# Agentic AI Systems — Worklet 15
# Executive Decision Support System
## Lab 7 · Agentic Node Graph — explicit stages, branches and back-edges

**Phase B · Orchestration** &nbsp;|&nbsp; **Goal:** wire the specialist agents into an explicit node graph.

---

### What Lab 7 contributes to Worklet 15

| | |
|---|---|
| **Inputs** | The specialist agents from Labs 3–6. |
| **Outputs** | An explicit graph: stages, parallel branches, gates and back-edges. |
| **Agents / nodes** | Coordinator → 4 domain agents → KPI join → report → validation → human gate → publish. |
| **Deliverable** | A runnable node graph with conditional routing and a validation → regenerate loop. |
| **Acceptance** | The worklet moves from request to release through **explicit stages**, with the failure path wired in. |

### Why a graph and not the Lab 6 list

Lab 6's run plan was a *list* with dependency hints — the order lived in how the list was written. Here the **edges are data**, which buys three things a list cannot:

* the graph can be **drawn** (Cell 4 renders ASCII and Mermaid), so a reviewer checks the wiring without reading the engine,
* it can be **statically checked** — unreachable nodes and dangling edges are found *before* anything runs,
* and it can express a **back-edge**: a failure that sends control backwards without restarting the run.

> A real bug this caught: the fan-out to three of the four domain agents was implicit in the parallel group. `validate()` reported them **unreachable** until four real edges were added. That is the check earning its place.

### The run this notebook performs

The report node deliberately omits the source system for `KPI-R2` on its first pass — exactly what Chapter 15's validation gate exists to catch (*"every figure cites its source"*).

```
validation --fail--> regenerate --(back-edge)--> report_agent --> validation --pass--> human_gate
```

The back-edge fires once, the defect is repaired, and **the four domain agents do not re-run**. That is the whole reason the loop is drawn as an edge instead of restarting the graph.

### How to run

Top to bottom, **no API key and no network**. Cell 1 writes `edss_core.py` (Labs 1–2) and `edss_stack.py` (Labs 3–5). Cell 7 executes the rubric as assertions.

---

### Cell 1 · The accumulated stack

`edss_core.py` (Labs 1–2) and `edss_stack.py` (Labs 3–5), written to disk so this notebook runs standalone.

*(Collapse both — prior work.)*

In [1]:
%%writefile edss_core.py
"""edss_core — shared foundation for the Worklet 15 lab notebooks.

Worklet 15 · Executive Decision Support System · Agentic AI (26ECAC401)

This is the Lab 1 + Lab 2 foundation, unchanged: the institutional data, the
typed tool contracts, the single ToolRouter choke point, the Connector, and the
validation gate. Labs 3-8 build on top of it rather than restating it, so every
notebook works against the same tools and the same numbers.

Written to disk by the first code cell of each lab notebook, so each notebook
runs standalone in Colab with no setup.
"""

from __future__ import annotations

import csv
import io
import json
import math
import re
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

# 1 · Institutional data (identical to Lab 2, Cell 3)
# =============================================================================

REPORTING_PERIOD = "academic year 2025-26"

DEPARTMENT_REPORTS_CSV = """dept_code,dept_name,category,students_appeared,students_passed,co_po_attainment,sanctioned_faculty,publications,citations,grants_lakh,eligible_students,placed_students,total_package_lpa,interns_completed,budget_allocated_lakh,budget_spent_lakh,figure_status
CSE,Computer Science & Engineering,circuit,240,231,2.71,34,58,402,96.5,228,212,2247.2,205,420,393,final
ECE,Electronics & Communication,circuit,180,168,2.58,26,34,221,41.0,172,143,1058.2,149,310,276,final
ISE,Information Science & Engineering,circuit,120,116,2.66,18,21,140,22.5,114,103,947.6,96,240,221,final
EEE,Electrical & Electronics,circuit,120,109,2.41,17,16,88,18.0,116,86,524.6,84,210,181,final
MECH,Mechanical Engineering,core,180,158,2.33,24,19,95,27.5,174,111,599.4,102,300,247,final
CIVIL,Civil Engineering,core,120,101,2.19,16,11,47,12.0,118,62,285.2,61,190,149,final
CHEM,Chemical Engineering,core,60,54,2.28,11,14,76,15.5,58,33,168.3,34,140,118,final
BT,Biotechnology,core,60,55,2.35,10,12,63,9.5,57,29,142.1,30,130,102,final
"""

NUMERIC_COLUMNS = {
    "students_appeared": int, "students_passed": int, "co_po_attainment": float,
    "sanctioned_faculty": int, "publications": int, "citations": int,
    "grants_lakh": float, "eligible_students": int, "placed_students": int,
    "total_package_lpa": float, "interns_completed": int,
    "budget_allocated_lakh": float, "budget_spent_lakh": float,
}

KPI_DEFINITIONS: Dict[str, Dict[str, Any]] = {
    "KPI-A1": {"name": "Institution pass percentage", "domain": "Academic",
               "method": "ratio", "numerator": "students_passed",
               "denominator": "students_appeared", "scale": 100, "unit": "%",
               "formula": "sum(students_passed) / sum(students_appeared) x 100",
               "target": 92.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Academic ERP"},
    "KPI-A2": {"name": "CO/PO attainment index", "domain": "Academic",
               "method": "weighted_mean", "value_field": "co_po_attainment",
               "weight_field": "students_appeared", "unit": "index 0-3",
               "formula": "sum(co_po_attainment x students_appeared) / sum(students_appeared)",
               "target": 2.50, "target_source": "NAAC criterion 2",
               "direction": "higher_is_better", "source_system": "Academic ERP (OBE module)"},
    "KPI-R1": {"name": "Publications per faculty", "domain": "Research",
               "method": "ratio", "numerator": "publications",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(publications) / sum(sanctioned_faculty)",
               "target": 1.50, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R2": {"name": "Citations per faculty", "domain": "Research",
               "method": "ratio", "numerator": "citations",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(citations) / sum(sanctioned_faculty)",
               "target": 9.00, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R3": {"name": "Grants sanctioned", "domain": "Research",
               "method": "sum", "value_field": "grants_lakh", "unit": "INR lakh",
               "formula": "sum(grants_lakh)",
               "target": 300.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-P1": {"name": "Placement percentage", "domain": "Placement",
               "method": "ratio", "numerator": "placed_students",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(placed_students) / sum(eligible_students) x 100",
               "target": 80.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P2": {"name": "Average package", "domain": "Placement",
               "method": "ratio", "numerator": "total_package_lpa",
               "denominator": "placed_students", "scale": 1, "unit": "LPA",
               "formula": "sum(total_package_lpa) / sum(placed_students)",
               "target": 8.00, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P3": {"name": "Internship completion", "domain": "Placement",
               "method": "ratio", "numerator": "interns_completed",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(interns_completed) / sum(eligible_students) x 100",
               "target": 85.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-F1": {"name": "Budget utilisation", "domain": "Finance",
               "method": "ratio", "numerator": "budget_spent_lakh",
               "denominator": "budget_allocated_lakh", "scale": 100, "unit": "%",
               "formula": "sum(budget_spent_lakh) / sum(budget_allocated_lakh) x 100",
               "target": 90.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Finance & Accounts ledger",
               "figure_status": "provisional",
               "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                         "books in August."},
}

BENCHMARK_API_DB = {
    "KPI-R1": {"peer_median": 1.82, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-R2": {"peer_median": 11.40, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-P1": {"peer_median": 78.60, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
}

DOMAINS = ["Academic", "Research", "Placement", "Finance"]
DOMAIN_KPIS = {d: [k for k, v in KPI_DEFINITIONS.items() if v["domain"] == d] for d in DOMAINS}

# Registrar-authorised read access, per Gate 1 of the hybrid execution methodology.
ACCESS_REGISTER = {
    "Academic":  {"source_system": "Academic ERP — examination & OBE records",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["pass percentage", "CO/PO attainment", "curriculum compliance"]},
    "Research":  {"source_system": "Research & Publications repository (Scopus-synced)",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["publications", "citations", "grants sanctioned"]},
    "Placement": {"source_system": "Placement Cell tracker",
                  "read_access": "granted by Registrar on 2026-06-04", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["placement percentage", "average package",
                              "internship completion"],
                  "caveat": "3 provisional offers are excluded until the Placement Cell "
                            "verifies them."},
    "Finance":   {"source_system": "Finance & Accounts ledger",
                  "read_access": "granted by Registrar on 2026-06-07",
                  "figure_status": "provisional",
                  # 2025-26 is deliberately absent: the books are not closed, and the
                  # Lab 1 planner must notice that rather than plan over it.
                  "periods": ["2023-24", "2024-25"],
                  "metrics": ["budget utilisation", "revenue", "grant funds"],
                  "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                            "books in August."},
}


# =============================================================================
# 2 · Tool contracts (identical to Lab 2, Cell 4)
# =============================================================================

TOOL_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "read_department_reports": {
        "description": "Read per-department metric rows for one reporting period.",
        "parameters": {"type": "object", "properties": {
            "period": {"type": "string"},
            "departments": {"type": "array", "items": {"type": "string"}},
            "category": {"type": "string", "enum": ["circuit", "core"]}},
            "required": ["period"]},
        "returns": "{rows, row_count, source_system} | {error}"},
    "read_kpi_definitions": {
        "description": "Read approved KPI definitions: formula, unit, target, authority.",
        "parameters": {"type": "object", "properties": {
            "kpi_ids": {"type": "array", "items": {"type": "string"}}}, "required": []},
        "returns": "{definitions, count} | {error}"},
    "read_external_benchmark": {
        "description": "Published peer median for a KPI. Coverage is partial.",
        "parameters": {"type": "object", "properties": {"kpi_id": {"type": "string"}},
                       "required": ["kpi_id"]},
        "returns": "{kpi_id, peer_median, cohort, year} | {error}"},
    "filter_rows": {
        "description": "Filter department rows by an exact field value; errors on empty.",
        "parameters": {"type": "object", "properties": {
            "rows": {"type": "array", "items": {"type": "object"}},
            "field": {"type": "string"}, "value": {"type": "string"}},
            "required": ["rows", "field", "value"]},
        "returns": "{rows, row_count, filter} | {error}"},
    "compute_kpi": {
        "description": "Compute one approved KPI over department rows, with evidence.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"},
            "rows": {"type": "array", "items": {"type": "object"}}},
            "required": ["kpi_id", "rows"]},
        "returns": "{kpi_id, value, unit, formula, evidence, source_system} | {error}"},
    "compare_to_target": {
        "description": "Compare a KPI value to its approved target; flags rounding risk.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"}, "value": {"type": "number"}},
            "required": ["kpi_id", "value"]},
        "returns": "{value, target, variance, status, rounding_warning} | {error}"},
    "compute_variance": {
        "description": "Difference between a KPI value and any reference figure.",
        "parameters": {"type": "object", "properties": {
            "value": {"type": "number"}, "reference": {"type": "number"},
            "label": {"type": "string"}}, "required": ["value", "reference", "label"]},
        "returns": "{variance, abs_variance, pct_of_reference} | {error}"},
    "draft_notification": {
        "description": "DRAFT a leadership notification. Never sends. Sources required.",
        "parameters": {"type": "object", "properties": {
            "audience": {"type": "string"}, "subject": {"type": "string"},
            "findings": {"type": "array", "items": {"type": "object"}}},
            "required": ["audience", "subject", "findings"]},
        "returns": "{status: 'DRAFT - NOT SENT', body, requires_approval_from} | {error}"},
}


def _typed_rows(text: str) -> List[Dict[str, Any]]:
    rows = []
    for raw in csv.DictReader(io.StringIO(text)):
        row = dict(raw)
        for col, caster in NUMERIC_COLUMNS.items():
            if col in row:
                try:
                    row[col] = caster(row[col])
                except (TypeError, ValueError):
                    return [{"error": f"column '{col}' is not {caster.__name__}"}]
        rows.append(row)
    return rows


# Optional demo latency, so the parallel/sequential distinction in the run
# timeline is visible. Off by default; never affects any computed figure.
SOURCE_LATENCY_SECONDS = 0.0


def read_department_reports(period: str, departments=None, category=None) -> Dict[str, Any]:
    if period != REPORTING_PERIOD:
        return {"error": f"no department report export for '{period}'; the only loaded "
                         f"period is '{REPORTING_PERIOD}'"}
    if SOURCE_LATENCY_SECONDS:
        time.sleep(SOURCE_LATENCY_SECONDS)
    rows = _typed_rows(DEPARTMENT_REPORTS_CSV)
    if rows and "error" in rows[0]:
        return {"error": rows[0]["error"]}
    if departments:
        known = {r["dept_code"] for r in rows}
        unknown = [d for d in departments if d not in known]
        if unknown:
            return {"error": f"unknown department code(s): {unknown}; known: {sorted(known)}"}
        rows = [r for r in rows if r["dept_code"] in departments]
    if category:
        rows = [r for r in rows if r["category"] == category]
        if not rows:
            return {"error": f"no departments in category '{category}'"}
    if not rows:
        return {"error": "empty department report export"}
    return {"rows": rows, "row_count": len(rows), "period": period,
            "source_system": "Department reports export (Registrar-authorised read)"}


def read_kpi_definitions(kpi_ids=None) -> Dict[str, Any]:
    if kpi_ids:
        unknown = [k for k in kpi_ids if k not in KPI_DEFINITIONS]
        if unknown:
            return {"error": f"unknown KPI id(s): {unknown}"}
        defs = {k: KPI_DEFINITIONS[k] for k in kpi_ids}
    else:
        defs = dict(KPI_DEFINITIONS)
    return {"definitions": defs, "count": len(defs),
            "source_system": "Institutional KPI register (approved definitions)"}


def read_external_benchmark(kpi_id: str) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    record = BENCHMARK_API_DB.get(kpi_id)
    if record is None:
        return {"error": f"no published external benchmark for {kpi_id} "
                         f"({KPI_DEFINITIONS[kpi_id]['name']}) in the benchmark API"}
    return {"kpi_id": kpi_id, **record, "source_system": "External benchmark API (NIRF cohort)"}


def filter_rows(rows, field: str, value: str) -> Dict[str, Any]:
    if not rows:
        return {"error": "filter_rows received an empty row set"}
    if field not in rows[0]:
        return {"error": f"unknown field '{field}'; available: {sorted(rows[0])}"}
    subset = [r for r in rows if str(r[field]) == str(value)]
    if not subset:
        return {"error": f"filter {field}={value!r} matched 0 of {len(rows)} rows — "
                         f"refusing to return an empty subset"}
    return {"rows": subset, "row_count": len(subset), "filter": f"{field}=={value}",
            "filtered_from": len(rows)}


def compute_kpi(kpi_id: str, rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Generic: the arithmetic comes from the KPI definition, not from this code."""
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    if not rows:
        return {"error": f"cannot compute {kpi_id} over an empty row set"}
    d = KPI_DEFINITIONS[kpi_id]
    method, evidence = d["method"], []
    try:
        if method == "ratio":
            num_f, den_f, scale = d["numerator"], d["denominator"], d["scale"]
            num = sum(r[num_f] for r in rows)
            den = sum(r[den_f] for r in rows)
            if den == 0:
                return {"error": f"{kpi_id}: denominator sum({den_f}) is zero"}
            value = num / den * scale
            inputs = {f"sum({num_f})": round(num, 4), f"sum({den_f})": round(den, 4)}
            for r in rows:
                dv = (r[num_f] / r[den_f] * scale) if r[den_f] else None
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[num_f], "denominator": r[den_f],
                                 "numerator_field": num_f, "denominator_field": den_f,
                                 "dept_value": round(dv, 2) if dv is not None else None,
                                 "share_of_denominator_pct": round(r[den_f] / den * 100, 2)})
        elif method == "weighted_mean":
            vf, wf = d["value_field"], d["weight_field"]
            wsum = sum(r[wf] for r in rows)
            if wsum == 0:
                return {"error": f"{kpi_id}: total weight sum({wf}) is zero"}
            value = sum(r[vf] * r[wf] for r in rows) / wsum
            inputs = {f"sum({vf} x {wf})": round(sum(r[vf] * r[wf] for r in rows), 4),
                      f"sum({wf})": wsum}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": r[wf],
                                 "numerator_field": vf, "denominator_field": wf,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[wf] / wsum * 100, 2)})
        elif method == "sum":
            vf = d["value_field"]
            value = sum(r[vf] for r in rows)
            inputs = {f"sum({vf})": round(value, 4)}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": None,
                                 "numerator_field": vf, "denominator_field": None,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[vf] / value * 100, 2)
                                 if value else None})
        else:
            return {"error": f"unsupported KPI method '{method}' for {kpi_id}"}
    except KeyError as missing:
        return {"error": f"{kpi_id}: department rows are missing column {missing}"}

    statuses = {r.get("figure_status", "final") for r in rows}
    figure_status = d.get("figure_status") or ("final" if statuses == {"final"} else "mixed")
    out = {"kpi_id": kpi_id, "name": d["name"], "domain": d["domain"],
           "value": round(value, 4), "unit": d["unit"], "formula": d["formula"],
           "method": method, "n_departments": len(rows),
           "departments": [r["dept_code"] for r in rows],
           "inputs_used": inputs, "evidence": evidence,
           "source_system": d["source_system"], "figure_status": figure_status}
    if d.get("caveat"):
        out["caveat"] = d["caveat"]
    return out


def compare_to_target(kpi_id: str, value: float) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    d = KPI_DEFINITIONS[kpi_id]
    target = d["target"]
    variance = value - target
    pct_of_target = value / target * 100 if target else None
    gap_pct = abs(variance) / target * 100 if target else 0
    status = "on target" if variance >= 0 else ("at risk" if gap_pct <= 2.0 else "below target")
    rounding_warning = None
    if variance < 0:
        for places in (0, 1):
            if round(value, places) >= target:
                rounding_warning = (f"{value:.4g} rounds to {round(value, places)} at "
                                    f"{places} decimal place(s), which would read as meeting "
                                    f"the target of {target}. Report the unrounded figure.")
                break
    return {"kpi_id": kpi_id, "name": d["name"], "value": round(value, 4), "target": target,
            "target_source": d["target_source"], "unit": d["unit"],
            "variance": round(variance, 4),
            "pct_of_target": round(pct_of_target, 2) if pct_of_target else None,
            "status": status, "rounding_warning": rounding_warning}


def compute_variance(value: float, reference: float, label: str) -> Dict[str, Any]:
    if reference == 0:
        return {"error": f"cannot compute variance against a zero reference ({label})"}
    variance = value - reference
    return {"value": round(value, 4), "reference": round(reference, 4), "label": label,
            "variance": round(variance, 4), "abs_variance": round(abs(variance), 4),
            "pct_of_reference": round(value / reference * 100, 2)}


def draft_notification(audience: str, subject: str, findings: List[Dict[str, Any]]):
    if not findings:
        return {"error": "refusing to draft a notification with no findings"}
    unsourced = [f for f in findings if not f.get("source")]
    if unsourced:
        return {"error": f"{len(unsourced)} finding(s) carry no source; every figure in a "
                         f"leadership notification must cite where it came from"}
    body = "\n".join(f"  {i}. {f['statement']}\n     source: {f['source']}"
                     for i, f in enumerate(findings, 1))
    return {"status": "DRAFT - NOT SENT", "audience": audience, "subject": subject,
            "body": body, "findings_cited": len(findings),
            "requires_approval_from": "Principal / Registrar (Chapter 15 human gate)",
            "note": "This tool cannot send. Circulation happens only after human approval."}


TOOLS = {
    "read_department_reports": read_department_reports,
    "read_kpi_definitions": read_kpi_definitions,
    "read_external_benchmark": read_external_benchmark,
    "filter_rows": filter_rows,
    "compute_kpi": compute_kpi,
    "compare_to_target": compare_to_target,
    "compute_variance": compute_variance,
    "draft_notification": draft_notification,
}


# =============================================================================
# 3 · Router, connector, validation gate (Lab 2, Cells 5-6)
# =============================================================================

class ToolContractError(Exception):
    """Raised before a call when arguments do not satisfy the tool schema."""


class ToolError(Exception):
    """Raised after a call when the tool returned an explicit error."""


_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


def _summarise(obj, _depth=0):
    if isinstance(obj, list):
        if len(obj) > 3 and all(isinstance(x, dict) for x in obj):
            return f"[{len(obj)} rows: {', '.join(str(x.get('dept_code', '?')) for x in obj)}]"
        return [_summarise(x, _depth + 1) for x in obj]
    if isinstance(obj, dict):
        return {k: _summarise(v, _depth + 1) for k, v in obj.items()}
    if isinstance(obj, str) and len(obj) > 160:
        return obj[:157] + "..."
    return obj


class ToolRouter:
    """The single choke point. Validates, executes, logs, raises on error."""

    def __init__(self, agent: str = "Coordinator"):
        self.agent = agent
        self.trace: List[Dict[str, Any]] = []
        self.raw_outputs: List[Any] = []

    def validate_input(self, tool: str, kwargs: Dict[str, Any]) -> None:
        if tool not in TOOL_SCHEMAS:
            raise ToolContractError(f"no such tool: '{tool}'")
        schema = TOOL_SCHEMAS[tool]["parameters"]
        props, required = schema["properties"], schema.get("required", [])
        for name in required:
            if name not in kwargs:
                raise ToolContractError(f"{tool}: missing required argument '{name}'")
        for name, val in kwargs.items():
            if name not in props:
                raise ToolContractError(
                    f"{tool}: unknown argument '{name}'; accepts {sorted(props)}")
            spec = props[name]
            expected = _JSON_TYPES.get(spec["type"])
            if expected and not isinstance(val, expected):
                raise ToolContractError(
                    f"{tool}.{name}: expected {spec['type']}, got {type(val).__name__}")
            if spec.get("enum") and val not in spec["enum"]:
                raise ToolContractError(f"{tool}.{name}: {val!r} not in {spec['enum']}")

    def call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        self.validate_input(tool, kwargs)
        started = time.perf_counter()
        result = TOOLS[tool](**kwargs)
        entry = {"step": len(self.trace) + 1, "agent": self.agent, "tool": tool,
                 "reason": reason, "input": _summarise(kwargs), "output": _summarise(result),
                 "ok": "error" not in result,
                 "ms": round((time.perf_counter() - started) * 1000, 2),
                 "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self.trace.append(entry)
        self.raw_outputs.append(result)
        if "error" in result:
            raise ToolError(f"{tool} failed: {result['error']}")
        return result

    def try_call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        try:
            return self.call(tool, reason, **kwargs)
        except ToolError:
            return dict(TOOLS[tool](**kwargs))


class Connector:
    """One governed surface over every source. No agent touches a source directly."""

    RESOURCES = {
        "department reports":              "read_department_reports",
        "institutional KPIs & dashboards": "read_kpi_definitions",
        "external benchmarks":             "read_external_benchmark",
    }

    def __init__(self, router: ToolRouter):
        self.router = router

    def read(self, resource: str, reason: str, required: bool = True, **params):
        if resource not in self.RESOURCES:
            raise ToolContractError(f"connector exposes no resource '{resource}'; "
                                    f"available: {sorted(self.RESOURCES)}")
        caller = self.router.call if required else self.router.try_call
        return caller(self.RESOURCES[resource], reason, **params)


class Validated:
    def __init__(self, checks: List[tuple], subject: str):
        self.checks, self.subject = checks, subject
        self.failing = [name for name, ok, _ in checks if not ok]
        self.valid = not self.failing

    def __repr__(self):
        return f"<Validated {self.subject}: {'VALID' if self.valid else self.failing}>"


def validate_kpi_result(computed: Dict[str, Any],
                        compared: Optional[Dict[str, Any]] = None) -> Validated:
    ev = computed.get("evidence", [])
    checks = [
        ("value is a finite number",
         isinstance(computed.get("value"), (int, float)) and math.isfinite(computed["value"]),
         f"value={computed.get('value')}"),
        ("the formula used is recorded", bool(computed.get("formula")),
         computed.get("formula", "")),
        ("the figure cites its source system", bool(computed.get("source_system")),
         computed.get("source_system", "")),
        ("raw inputs are recorded", bool(computed.get("inputs_used")),
         str(computed.get("inputs_used", ""))),
        ("every department in the evidence is identified",
         bool(ev) and all(e.get("dept_code") for e in ev), f"{len(ev)} evidence rows"),
        ("evidence covers exactly the departments computed over",
         len(ev) == computed.get("n_departments"), f"{len(ev)} rows"),
        ("provisional figures carry a caveat",
         computed.get("figure_status") == "final" or bool(computed.get("caveat")),
         f"figure_status={computed.get('figure_status')}"),
    ]
    if compared is not None:
        checks.append(("the target names the authority that set it",
                       bool(compared.get("target_source")), compared.get("target_source", "")))
    return Validated(checks, computed.get("kpi_id", "?"))


# =============================================================================


Overwriting edss_core.py


In [2]:
%%writefile edss_stack.py
"""edss_stack — the capabilities Labs 3, 4 and 5 added, in one importable layer.

Worklet 15 · Executive Decision Support System · Agentic AI (26ECAC401)

    Lab 3  Skill            reusable, schema-typed capability packages
    Lab 4  RunState/Archive short-term run state vs the long-term baseline,
                            with tag and TF-IDF-similarity retrieval
    Lab 5  MCPServer/Client one typed, scoped connector over file + SQLite + API

Labs 6-8 bind these together rather than restating them: Lab 6 puts a runtime
around them, Lab 7 wires them into a node graph, Lab 8 runs them in parallel.

Written to disk by the first code cells of Labs 6-8 so each notebook is
standalone.
"""

import csv
import io
import json
import math
import re
import sqlite3
import textwrap
import time
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

from edss_core import (DOMAINS, KPI_DEFINITIONS, REPORTING_PERIOD, ACCESS_REGISTER,
                       DEPARTMENT_REPORTS_CSV, NUMERIC_COLUMNS, BENCHMARK_API_DB,
                       ToolRouter, Connector, ToolError, ToolContractError,
                       compute_kpi, compare_to_target, validate_kpi_result,
                       _typed_rows)

_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


# ==========================================================================
# LAB 3 — the Skill framework
# ==========================================================================

class SkillContractError(Exception):
    """Raised when a skill's input or output violates its declared schema."""


@dataclass
class Skill:
    """A reusable capability package.

    name          stable identifier
    description   what it does, in one line
    instructions  the procedure it follows — the part a human or a model reads
    constraints   what it must never do
    input_schema  typed contract, validated BEFORE the skill runs
    output_schema typed contract, validated AFTER it runs
    tools         the tool names it is permitted to call
    """
    name: str
    description: str
    instructions: str
    constraints: List[str]
    input_schema: Dict[str, Any]
    output_schema: Dict[str, Any]
    tools: List[str]
    _fn: Optional[Callable] = None
    invocations: List[Dict[str, Any]] = field(default_factory=list)

    # ---- schema validation, both directions ----------------------------
    def _validate(self, payload: Dict[str, Any], schema: Dict[str, Any],
                  where: str) -> None:
        props, required = schema.get("properties", {}), schema.get("required", [])
        for name in required:
            if name not in payload:
                raise SkillContractError(
                    f"{self.name} {where}: missing required field '{name}'")
        for name, value in payload.items():
            if name not in props:
                raise SkillContractError(
                    f"{self.name} {where}: unexpected field '{name}'; "
                    f"accepts {sorted(props)}")
            expected = _JSON_TYPES.get(props[name]["type"])
            if expected and not isinstance(value, expected):
                raise SkillContractError(
                    f"{self.name} {where}.{name}: expected {props[name]['type']}, "
                    f"got {type(value).__name__}")
            if props[name].get("enum") and value not in props[name]["enum"]:
                raise SkillContractError(
                    f"{self.name} {where}.{name}: {value!r} not in "
                    f"{props[name]['enum']}")

    def run(self, router: "ToolRouter", **inputs) -> Dict[str, Any]:
        self._validate(inputs, self.input_schema, "input")
        before = len(router.trace)
        result = self._fn(router, **inputs)
        self._validate(result, self.output_schema, "output")
        used = [e["tool"] for e in router.trace[before:]]
        stray = [t for t in used if t not in self.tools]
        if stray:
            raise SkillContractError(
                f"{self.name} called tools outside its declared surface: {stray}")
        self.invocations.append({"inputs": inputs, "tools_used": used,
                                 "output_keys": sorted(result)})
        return result

    def card(self) -> str:
        return (f"SKILL  {self.name}\n"
                f"  {self.description}\n"
                f"  inputs   : {list(self.input_schema.get('properties', {}))}"
                f"  (required: {self.input_schema.get('required', [])})\n"
                f"  outputs  : {list(self.output_schema.get('properties', {}))}\n"
                f"  tools    : {self.tools}\n"
                f"  constraints:\n"
                + "\n".join(f"    - {c}" for c in self.constraints))


SKILL_REGISTRY: Dict[str, Skill] = {}


def register(skill: Skill) -> Skill:
    SKILL_REGISTRY[skill.name] = skill
    return skill


# ==========================================================================
# LAB 3 — skill: review_domain
# ==========================================================================

REVIEW_DOMAIN = Skill(
    name="review_domain",
    description="Review one domain's approved KPIs against their targets, with "
                "per-department evidence.",
    instructions=(
        "1. Read the department rows for the requested period through the connector.\n"
        "2. If a department filter is supplied, narrow to it and refuse an empty subset.\n"
        "3. For every approved KPI belonging to the domain: read its definition, compute\n"
        "   it with the registered formula, and compare it to its approved target.\n"
        "4. Return the KPI set with variance, status, source system and figure status.\n"
        "5. Report which KPIs are off target; never recommend an action — that is the\n"
        "   Report Generation Agent's job, behind the human gate."),
    constraints=[
        "Use only approved KPI definitions; never invent a formula.",
        "Never average department percentages — aggregate raw counts.",
        "Label any figure that is not final; never present it as an actual.",
        "Emit no recommendation and take no action.",
    ],
    input_schema={
        "type": "object",
        "properties": {
            "domain": {"type": "string", "enum": DOMAINS},
            "period": {"type": "string"},
            "category": {"type": "string", "enum": ["circuit", "core"]},
        },
        "required": ["domain", "period"],
    },
    output_schema={
        "type": "object",
        "properties": {
            "domain": {"type": "string"},
            "period": {"type": "string"},
            "scope": {"type": "string"},
            "kpis": {"type": "array"},
            "off_target": {"type": "array"},
            "headline": {"type": "string"},
            "provisional": {"type": "boolean"},
        },
        "required": ["domain", "period", "scope", "kpis", "off_target", "headline",
                     "provisional"],
    },
    tools=["read_department_reports", "filter_rows", "read_kpi_definitions",
           "compute_kpi", "compare_to_target"],
)


def _review_domain(router, domain, period, category=None):
    connector = Connector(router)
    rows = connector.read("department reports",
                          reason=f"review_domain({domain}): per-department metrics",
                          period=period)["rows"]
    scope = "all departments"
    if category:
        rows = router.call("filter_rows",
                           reason=f"review_domain({domain}): narrow to {category} branches",
                           rows=rows, field="category", value=category)["rows"]
        scope = f"{category} departments"

    kpi_ids = [k for k, d in KPI_DEFINITIONS.items() if d["domain"] == domain]
    kpis, provisional = [], False
    for kpi_id in kpi_ids:
        connector.read("institutional KPIs & dashboards",
                       reason=f"{kpi_id}: approved formula and target",
                       kpi_ids=[kpi_id])
        computed = router.call("compute_kpi", reason=f"{kpi_id}: apply the formula",
                               kpi_id=kpi_id, rows=rows)
        compared = router.call("compare_to_target", reason=f"{kpi_id}: variance vs target",
                               kpi_id=kpi_id, value=computed["value"])
        gate = validate_kpi_result(computed, compared)
        if not gate.valid:
            raise ToolError(f"validation gate failed for {kpi_id}: {gate.failing}")
        provisional = provisional or computed["figure_status"] != "final"
        kpis.append({
            "kpi_id": kpi_id, "name": computed["name"], "value": computed["value"],
            "unit": computed["unit"], "target": compared["target"],
            "target_source": compared["target_source"], "variance": compared["variance"],
            "status": compared["status"], "formula": computed["formula"],
            "source_system": computed["source_system"],
            "figure_status": computed["figure_status"],
            "rounding_warning": compared["rounding_warning"],
            "evidence": computed["evidence"],
        })

    off = [k["kpi_id"] for k in kpis if k["status"] != "on target"]
    headline = (f"{domain}: {len(off)} of {len(kpis)} KPIs off target across {scope} "
                f"for {period}." + (" Figures include a provisional source."
                                    if provisional else ""))
    return {"domain": domain, "period": period, "scope": scope, "kpis": kpis,
            "off_target": off, "headline": headline, "provisional": provisional}


REVIEW_DOMAIN._fn = _review_domain
register(REVIEW_DOMAIN)


# ==========================================================================
# LAB 3 — skill: format_document
# ==========================================================================

FORMAT_DOCUMENT = Skill(
    name="format_document",
    description="Render a titled, sectioned payload as a leadership-ready document "
                "in a chosen template.",
    instructions=(
        "1. Emit the title, then the metadata block as key: value lines.\n"
        "2. Emit each section in order: a heading, then its bullets, then its table\n"
        "   if it has one.\n"
        "3. In 'brief' template, number the sections; in 'memo' template, do not.\n"
        "4. Append the source line verbatim if one is supplied.\n"
        "5. Reproduce every value exactly as given — never reformat a number."),
    constraints=[
        "Never compute, round or reword a value; this skill only lays out.",
        "Never drop a section or a caveat that was passed in.",
        "Stay domain-agnostic: no KPI, department or worklet logic here.",
    ],
    input_schema={
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "meta": {"type": "object"},
            "sections": {"type": "array"},
            "template": {"type": "string", "enum": ["brief", "memo"]},
            "source_line": {"type": "string"},
        },
        "required": ["title", "sections"],
    },
    output_schema={
        "type": "object",
        "properties": {"document": {"type": "string"},
                       "template": {"type": "string"},
                       "section_count": {"type": "integer"},
                       "line_count": {"type": "integer"}},
        "required": ["document", "template", "section_count", "line_count"],
    },
    tools=[],          # a pure formatting skill: it calls no tool at all
)


def _format_document(router, title, sections, meta=None, template="brief",
                     source_line=None):
    out = [title, "=" * len(title)]
    for k, v in (meta or {}).items():
        out.append(f"{k}: {v}")
    out.append("")
    for i, sec in enumerate(sections, 1):
        heading = sec.get("heading", "")
        out.append(f"{i}. {heading}" if template == "brief" else heading.upper())
        out.append("-" * 70)
        for b in sec.get("bullets", []):
            out.append(textwrap.fill(f"  - {b}", 88, subsequent_indent="    "))
        table = sec.get("table")
        if table:
            widths = [max(len(str(r[c])) for r in [dict(zip(table["columns"],
                                                            table["columns"]))] + table["rows"])
                      for c in table["columns"]]
            out.append("  " + "  ".join(str(c).ljust(w) for c, w in
                                        zip(table["columns"], widths)))
            for row in table["rows"]:
                out.append("  " + "  ".join(str(row[c]).ljust(w) for c, w in
                                            zip(table["columns"], widths)))
        out.append("")
    if source_line:
        out.append(source_line)
    document = "\n".join(out)
    return {"document": document, "template": template,
            "section_count": len(sections), "line_count": len(document.splitlines())}


FORMAT_DOCUMENT._fn = _format_document
register(FORMAT_DOCUMENT)


# ==========================================================================
# LAB 4 — the memory split
# ==========================================================================

class ArchiveImmutableError(Exception):
    """Raised on any attempt to edit a record that is already archived."""


@dataclass
class RunState:
    """Short-term working memory. One run. Thrown away at the end."""
    run_id: str
    period: str
    draft: Dict[str, Any] = field(default_factory=dict)
    scratch: Dict[str, Any] = field(default_factory=dict)
    retrieved: List[Dict[str, Any]] = field(default_factory=list)

    def note(self, key: str, value: Any) -> None:
        self.scratch[key] = value

    def summary(self) -> str:
        return (f"RunState({self.run_id}) draft_keys={sorted(self.draft)} "
                f"scratch_keys={sorted(self.scratch)} retrieved={len(self.retrieved)}")


class Archive:
    """Long-term store. Append-only; every write is versioned and timestamped."""

    def __init__(self):
        self._records: Dict[str, Dict[str, Any]] = {}
        self.writes: List[Dict[str, Any]] = []

    # ---- writing -------------------------------------------------------
    def put(self, record_id: str, record: Dict[str, Any]) -> Dict[str, Any]:
        if record_id in self._records:
            raise ArchiveImmutableError(
                f"'{record_id}' is already archived; the archive is append-only. "
                f"Write a new version id instead of editing the baseline.")
        stored = {**record, "record_id": record_id,
                  "archived_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self._records[record_id] = stored
        self.writes.append({"record_id": record_id, "tags": record.get("tags", []),
                            "at": stored["archived_at"]})
        return stored

    # ---- reading -------------------------------------------------------
    def get(self, record_id: str) -> Dict[str, Any]:
        # A COPY, so a caller mutating what it read cannot corrupt the baseline.
        return json.loads(json.dumps(self._records[record_id]))

    def all(self) -> List[Dict[str, Any]]:
        return [self.get(rid) for rid in self._records]

    def __len__(self):
        return len(self._records)

    def __contains__(self, record_id):
        return record_id in self._records


# ==========================================================================
# LAB 4 — what is in the archive
# ==========================================================================

archive = Archive()

PRIOR_BRIEFS = [
    {"record_id": "BRIEF-2023-24-Q4",
     "type": "decision_brief", "period": "2023-24",
     "title": "Placement recovery after the 2023 hiring slowdown",
     "tags": ["placement", "core-branches", "recruiter-outreach", "below-target"],
     "summary": "Placement closed at 71.80% against an 80% target. Core branches drove "
                "the gap; circuit branches held. The Council approved a recruiter "
                "outreach drive and deferred the package target.",
     "decisions": ["Approved INR 12 lakh recruiter outreach budget",
                   "Deferred the average-package target by one year"],
     "outcome": "Placement rose to 73.50% the following year — short of target but "
                "the trend reversed. The wider recruiter pool brought in lower-paying "
                "offers, and average package began to slide."},
    {"record_id": "BRIEF-2024-25-Q2",
     "type": "decision_brief", "period": "2024-25",
     "title": "Research output against the NIRF submission",
     "tags": ["research", "publications", "citations", "nirf", "below-target"],
     "summary": "Publications per faculty stood at 1.05 against a 1.50 target and "
                "citations per faculty at 6.40. The Council approved a seed-grant "
                "scheme and asked for a per-department breakdown each quarter.",
     "decisions": ["Approved a seed-grant scheme for research-active faculty",
                   "Made department-level drill-down mandatory for research KPIs"],
     "outcome": "Publications rose to 1.12; the reporting change stuck and is now "
                "standing practice."},
    {"record_id": "BRIEF-2024-25-Q4",
     "type": "decision_brief", "period": "2024-25",
     "title": "Budget utilisation and the unreconciled Q4 position",
     "tags": ["finance", "budget-utilisation", "provisional", "reconciliation"],
     "summary": "Utilisation was reported at 88.10% but Q4 was unreconciled at the time "
                "of review. The Council declined to act on a provisional figure and "
                "deferred the decision to the post-audit cycle.",
     "decisions": ["Deferred all budget reallocation until the books closed",
                   "Required provisional figures to be labelled on the face of the brief"],
     "outcome": "Final utilisation came in at 86.90% — 1.2 points below the provisional "
                "figure, vindicating the deferral."},
    {"record_id": "BRIEF-2024-25-Q3",
     "type": "decision_brief", "period": "2024-25",
     "title": "Pass percentage and the rounding dispute",
     "tags": ["academic", "pass-percentage", "rounding", "at-risk"],
     "summary": "Pass percentage was circulated as '92%' when the unrounded figure was "
                "91.60%, which is below the 92.00% target. The Council directed that "
                "KPI figures be reported unrounded.",
     "decisions": ["Directed that all KPI figures be reported unrounded",
                   "Ordered remedial coaching in the two weakest departments"],
     "outcome": "The reporting rule is now enforced by the KPI engine itself."},
]

PRIOR_KPI_SNAPSHOTS = [
    {"record_id": "KPI-SNAPSHOT-2023-24", "type": "kpi_snapshot", "period": "2023-24",
     "title": "Institution KPI snapshot 2023-24",
     "tags": ["kpi-snapshot", "closed-year", "baseline"],
     "values": {"KPI-A1": 90.40, "KPI-A2": 2.38, "KPI-R1": 0.98, "KPI-R2": 5.90,
                "KPI-R3": 198.0, "KPI-P1": 71.80, "KPI-P2": 7.10, "KPI-P3": 64.20,
                "KPI-F1": 84.30}},
    {"record_id": "KPI-SNAPSHOT-2024-25", "type": "kpi_snapshot", "period": "2024-25",
     "title": "Institution KPI snapshot 2024-25",
     "tags": ["kpi-snapshot", "closed-year", "baseline"],
     "values": {"KPI-A1": 91.20, "KPI-A2": 2.44, "KPI-R1": 1.12, "KPI-R2": 6.40,
                "KPI-R3": 221.5, "KPI-P1": 73.50, "KPI-P2": 7.95, "KPI-P3": 69.10,
                "KPI-F1": 86.90}},
]

for rec in PRIOR_BRIEFS + PRIOR_KPI_SNAPSHOTS:
    archive.put(rec["record_id"], {k: v for k, v in rec.items() if k != "record_id"})


# ==========================================================================
# LAB 4 — retrieval by tag
# ==========================================================================

def retrieve_by_tags(store: Archive, tags: List[str], mode: str = "any",
                     record_type: Optional[str] = None) -> List[Dict[str, Any]]:
    """mode='any' -> union; mode='all' -> records carrying every tag."""
    if mode not in ("any", "all"):
        raise ValueError("mode must be 'any' or 'all'")
    wanted = {t.lower() for t in tags}
    hits = []
    for rec in store.all():
        if record_type and rec.get("type") != record_type:
            continue
        have = {t.lower() for t in rec.get("tags", [])}
        overlap = wanted & have
        if (mode == "any" and overlap) or (mode == "all" and wanted <= have):
            hits.append({**rec, "matched_tags": sorted(overlap),
                         "tag_score": len(overlap) / len(wanted)})
    return sorted(hits, key=lambda r: -r["tag_score"])


# ==========================================================================
# LAB 4 — retrieval by similarity
# ==========================================================================

STOPWORDS = {"the", "a", "an", "and", "or", "of", "to", "in", "on", "at", "for", "was",
             "were", "is", "are", "be", "been", "by", "with", "that", "this", "it",
             "as", "from", "but", "not", "no", "we", "our", "its", "their", "has",
             "had", "have", "than", "then", "so", "which", "when", "all", "one"}


def tokenize(text: str) -> List[str]:
    return [t for t in re.findall(r"[a-z0-9\-]+", text.lower())
            if t not in STOPWORDS and len(t) > 2]


class SimilarityIndex:
    """TF-IDF vectors + cosine similarity over the archive's text."""

    def __init__(self, records: List[Dict[str, Any]]):
        self.records = records
        self.docs = [self._text(r) for r in records]
        self.tokens = [tokenize(d) for d in self.docs]
        n = len(self.tokens)
        df = Counter(t for toks in self.tokens for t in set(toks))
        # Smoothed IDF, so a term in every document still carries a little weight.
        self.idf = {t: math.log((n + 1) / (c + 1)) + 1.0 for t, c in df.items()}
        self.vectors = [self._vector(toks) for toks in self.tokens]

    @staticmethod
    def _text(rec: Dict[str, Any]) -> str:
        parts = [rec.get("title", ""), rec.get("summary", ""),
                 " ".join(rec.get("tags", [])), " ".join(rec.get("decisions", [])),
                 rec.get("outcome", "")]
        return " ".join(p for p in parts if p)

    def _vector(self, toks: List[str]) -> Dict[str, float]:
        if not toks:
            return {}
        tf = Counter(toks)
        vec = {t: (c / len(toks)) * self.idf.get(t, 1.0) for t, c in tf.items()}
        norm = math.sqrt(sum(v * v for v in vec.values())) or 1.0
        return {t: v / norm for t, v in vec.items()}

    @staticmethod
    def _cosine(a: Dict[str, float], b: Dict[str, float]) -> float:
        small, large = (a, b) if len(a) < len(b) else (b, a)
        return sum(v * large.get(t, 0.0) for t, v in small.items())

    def query(self, text: str, k: int = 3,
              record_type: Optional[str] = None) -> List[Dict[str, Any]]:
        qv = self._vector(tokenize(text))
        scored = []
        for rec, vec in zip(self.records, self.vectors):
            if record_type and rec.get("type") != record_type:
                continue
            scored.append({**rec, "similarity": round(self._cosine(qv, vec), 4)})
        return sorted(scored, key=lambda r: -r["similarity"])[:k]


index = SimilarityIndex(archive.all())


# ==========================================================================
# LAB 4 — hybrid retrieval
# ==========================================================================

def retrieve(store: Archive, idx: SimilarityIndex, text: str,
             tags: Optional[List[str]] = None, k: int = 3,
             record_type: Optional[str] = None) -> List[Dict[str, Any]]:
    """Tag filter first (precision), then similarity rank (relevance)."""
    pool = (retrieve_by_tags(store, tags, "any", record_type) if tags
            else [r for r in store.all()
                  if not record_type or r.get("type") == record_type])
    if not pool:
        return []
    sub = SimilarityIndex(pool)
    hits = sub.query(text, k=k, record_type=record_type)
    by_id = {h["record_id"]: h for h in hits}
    for p in pool:
        if p["record_id"] in by_id and "tag_score" in p:
            by_id[p["record_id"]]["tag_score"] = p["tag_score"]
    return hits


# ==========================================================================
# LAB 5 — the MCP protocol
# ==========================================================================

# ---- error codes: one vocabulary for every backend ------------------------
E_UNKNOWN_RESOURCE = "UNKNOWN_RESOURCE"
E_UNAUTHORIZED     = "UNAUTHORIZED"
E_FORBIDDEN_SCOPE  = "FORBIDDEN_SCOPE"
E_BAD_REQUEST      = "BAD_REQUEST"
E_NOT_FOUND        = "NOT_FOUND"
E_BACKEND_FAILURE  = "BACKEND_FAILURE"


class MCPError(Exception):
    def __init__(self, code: str, message: str):
        super().__init__(f"{code}: {message}")
        self.code, self.message = code, message


@dataclass
class Resource:
    """One advertised capability of the server."""
    name: str
    description: str
    params: Dict[str, Any]          # typed parameter schema
    returns: str
    scope: str                      # the scope a token must hold
    backend: str                    # which backend currently serves it
    handler: Callable = None

    def manifest(self) -> Dict[str, Any]:
        return {"name": self.name, "description": self.description,
                "params": self.params, "returns": self.returns,
                "required_scope": self.scope, "served_by": self.backend}


class MCPServer:
    """One server, many backends. Agents never see which is which."""

    def __init__(self, name: str):
        self.name = name
        self.resources: Dict[str, Resource] = {}
        self.tokens: Dict[str, Dict[str, Any]] = {}
        self.log: List[Dict[str, Any]] = []

    # ---- registration & auth -------------------------------------------
    def expose(self, resource: Resource) -> None:
        self.resources[resource.name] = resource

    def issue_token(self, token: str, holder: str, scopes: List[str]) -> None:
        self.tokens[token] = {"holder": holder, "scopes": set(scopes)}

    def describe(self) -> Dict[str, Any]:
        """Capability discovery — what a client may call, without calling it."""
        return {"server": self.name,
                "resources": [r.manifest() for r in self.resources.values()]}

    # ---- the single entry point ----------------------------------------
    def call(self, token: str, resource: str, **params) -> Dict[str, Any]:
        started = time.perf_counter()
        entry = {"seq": len(self.log) + 1, "resource": resource,
                 "holder": self.tokens.get(token, {}).get("holder", "<unknown>"),
                 "params": {k: (f"<{len(v)} rows>" if isinstance(v, list) else v)
                            for k, v in params.items()},
                 "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}

        def finish(ok, code=None, note=""):
            entry.update({"ok": ok, "error_code": code, "note": note,
                          "ms": round((time.perf_counter() - started) * 1000, 2)})
            self.log.append(entry)

        if token not in self.tokens:
            finish(False, E_UNAUTHORIZED)
            raise MCPError(E_UNAUTHORIZED, "token not recognised by this server")
        if resource not in self.resources:
            finish(False, E_UNKNOWN_RESOURCE)
            raise MCPError(E_UNKNOWN_RESOURCE,
                           f"'{resource}' is not exposed; call describe() to see "
                           f"{sorted(self.resources)}")
        res = self.resources[resource]
        if res.scope not in self.tokens[token]["scopes"]:
            finish(False, E_FORBIDDEN_SCOPE)
            raise MCPError(E_FORBIDDEN_SCOPE,
                           f"'{resource}' needs scope '{res.scope}'; this token holds "
                           f"{sorted(self.tokens[token]['scopes'])}")
        for p, spec in res.params.items():
            if spec.get("required") and p not in params:
                finish(False, E_BAD_REQUEST)
                raise MCPError(E_BAD_REQUEST, f"{resource}: missing parameter '{p}'")
        try:
            data = res.handler(**params)
        except MCPError as exc:
            finish(False, exc.code, exc.message)
            raise
        except Exception as exc:
            finish(False, E_BACKEND_FAILURE, str(exc))
            raise MCPError(E_BACKEND_FAILURE, f"{res.backend} failed: {exc}")
        entry["served_by"] = res.backend
        finish(True)
        # The envelope is identical whatever the backend was.
        return {"ok": True, "resource": resource, "served_by": res.backend, "data": data,
                "meta": {"server": self.name, "at": entry["at"]}}


class MCPClient:
    """What an agent holds. It knows a token and resource names — nothing else."""

    def __init__(self, server: MCPServer, token: str):
        self._server, self._token = server, token

    def describe(self) -> Dict[str, Any]:
        return self._server.describe()

    def read(self, resource: str, **params) -> Any:
        return self._server.call(self._token, resource, **params)["data"]


# ==========================================================================
# LAB 5 — file, SQLite and API backends
# ==========================================================================

WORKDIR = Path("lab5_sources")
WORKDIR.mkdir(exist_ok=True)

# ---- backend 1: a FILE ----------------------------------------------------
CSV_PATH = WORKDIR / "department_reports_2025_26.csv"
CSV_PATH.write_text(DEPARTMENT_REPORTS_CSV)


def _cast(row: Dict[str, str]) -> Dict[str, Any]:
    out = dict(row)
    for col, caster in NUMERIC_COLUMNS.items():
        if col in out:
            out[col] = caster(out[col])
    return out


class FileBackend:
    name = "file:csv"

    def __init__(self, path: Path):
        self.path = path

    def department_reports(self, period: str, category: str = None) -> Dict[str, Any]:
        if period != REPORTING_PERIOD:
            raise MCPError(E_NOT_FOUND, f"no export on disk for '{period}'")
        with self.path.open() as fh:
            rows = [_cast(r) for r in csv.DictReader(fh)]
        if category:
            rows = [r for r in rows if r["category"] == category]
            if not rows:
                raise MCPError(E_NOT_FOUND, f"no departments in category '{category}'")
        return {"rows": rows, "row_count": len(rows), "period": period,
                "source_system": "Department reports export (CSV on disk)"}


# ---- backend 2: SQLite ----------------------------------------------------
DB_PATH = WORKDIR / "institution.db"
if DB_PATH.exists():
    DB_PATH.unlink()
conn = sqlite3.connect(DB_PATH)
conn.executescript("""
CREATE TABLE kpi_definition (
    kpi_id TEXT PRIMARY KEY, name TEXT, domain TEXT, unit TEXT, formula TEXT,
    target REAL, target_source TEXT, source_system TEXT, figure_status TEXT, caveat TEXT);
CREATE TABLE department_report (
    dept_code TEXT PRIMARY KEY, dept_name TEXT, category TEXT, period TEXT,
    payload TEXT);
""")
conn.executemany(
    "INSERT INTO kpi_definition VALUES (?,?,?,?,?,?,?,?,?,?)",
    [(k, d["name"], d["domain"], d["unit"], d["formula"], d["target"],
      d["target_source"], d["source_system"], d.get("figure_status", "final"),
      d.get("caveat")) for k, d in KPI_DEFINITIONS.items()])
conn.executemany(
    "INSERT INTO department_report VALUES (?,?,?,?,?)",
    [(r["dept_code"], r["dept_name"], r["category"], REPORTING_PERIOD, json.dumps(r))
     for r in (_cast(x) for x in csv.DictReader(io.StringIO(DEPARTMENT_REPORTS_CSV)))])
conn.commit()


class SQLiteBackend:
    name = "sqlite"

    def __init__(self, path: Path):
        self.path = path

    def _rows(self, sql: str, args=()) -> List[sqlite3.Row]:
        c = sqlite3.connect(self.path)
        c.row_factory = sqlite3.Row
        try:
            return c.execute(sql, args).fetchall()
        finally:
            c.close()

    def kpi_definitions(self, kpi_ids: List[str] = None) -> Dict[str, Any]:
        if kpi_ids:
            marks = ",".join("?" * len(kpi_ids))
            rows = self._rows(
                f"SELECT * FROM kpi_definition WHERE kpi_id IN ({marks})", kpi_ids)
            if len(rows) != len(kpi_ids):
                found = {r["kpi_id"] for r in rows}
                raise MCPError(E_NOT_FOUND,
                               f"unknown KPI id(s): {sorted(set(kpi_ids) - found)}")
        else:
            rows = self._rows("SELECT * FROM kpi_definition ORDER BY kpi_id")
        defs = {}
        for r in rows:
            d = dict(KPI_DEFINITIONS[r["kpi_id"]])   # formula fields for the calculator
            d.update({"name": r["name"], "domain": r["domain"], "target": r["target"],
                      "target_source": r["target_source"],
                      "source_system": r["source_system"]})
            defs[r["kpi_id"]] = d
        return {"definitions": defs, "count": len(defs),
                "source_system": "Institutional KPI register (SQLite)"}

    # The SAME resource the file backend serves — used in Cell 6 to swap sources.
    def department_reports(self, period: str, category: str = None) -> Dict[str, Any]:
        sql = "SELECT payload FROM department_report WHERE period = ?"
        args = [period]
        if category:
            sql += " AND category = ?"
            args.append(category)
        rows = [json.loads(r["payload"]) for r in self._rows(sql, args)]
        if not rows:
            raise MCPError(E_NOT_FOUND,
                           f"no rows for period '{period}'"
                           + (f" and category '{category}'" if category else ""))
        return {"rows": rows, "row_count": len(rows), "period": period,
                "source_system": "Department reports (SQLite table)"}


# ---- backend 3: a mock HTTP API -------------------------------------------
class APIBackend:
    name = "http:benchmark-api"
    LATENCY_S = 0.02

    def external_benchmark(self, kpi_id: str) -> Dict[str, Any]:
        time.sleep(self.LATENCY_S)              # a network call is not free
        if kpi_id not in KPI_DEFINITIONS:
            raise MCPError(E_BAD_REQUEST, f"'{kpi_id}' is not a known KPI id")
        record = BENCHMARK_API_DB.get(kpi_id)
        if record is None:
            raise MCPError(E_NOT_FOUND,
                           f"no published external benchmark for {kpi_id} "
                           f"({KPI_DEFINITIONS[kpi_id]['name']})")
        return {"kpi_id": kpi_id, **record,
                "source_system": "External benchmark API (NIRF cohort)"}


file_backend, sqlite_backend, api_backend = (FileBackend(CSV_PATH),
                                             SQLiteBackend(DB_PATH), APIBackend())


# ==========================================================================
# LAB 5 — expose resources, issue scoped tokens
# ==========================================================================

server = MCPServer("edss-connector")

server.expose(Resource(
    name="department_reports",
    description="Per-department metric rows for one reporting period.",
    params={"period": {"type": "string", "required": True},
            "category": {"type": "string", "required": False,
                         "enum": ["circuit", "core"]}},
    returns="{rows, row_count, period, source_system}",
    scope="read:departments", backend=file_backend.name,
    handler=file_backend.department_reports))

server.expose(Resource(
    name="kpi_definitions",
    description="Approved institution KPI definitions: formula, unit, target, authority.",
    params={"kpi_ids": {"type": "array", "required": False}},
    returns="{definitions, count, source_system}",
    scope="read:kpi-register", backend=sqlite_backend.name,
    handler=sqlite_backend.kpi_definitions))

server.expose(Resource(
    name="external_benchmark",
    description="Published peer median for a KPI. Coverage is partial.",
    params={"kpi_id": {"type": "string", "required": True}},
    returns="{kpi_id, peer_median, cohort, year, source_system}",
    scope="read:benchmarks", backend=api_backend.name,
    handler=api_backend.external_benchmark))

# ---- scoped tokens: the Registrar's authorisation, expressed as data -------
server.issue_token("tok-coordinator", "apniLeap AURA Coordinator",
                   ["read:departments", "read:kpi-register", "read:benchmarks"])
server.issue_token("tok-academic-agent", "Academic Agent",
                   ["read:departments", "read:kpi-register"])   # no benchmark access
server.issue_token("tok-intern", "Summer intern (read-only KPI register)",
                   ["read:kpi-register"])

# ==========================================================================
# A ready-made client for the coordinator
# ==========================================================================

coordinator_client = MCPClient(server, "tok-coordinator")


Overwriting edss_stack.py


### Cell 2 · Graph primitives

`Node`, `Edge`, parallel groups, conditional routers, back-edges, a static reachability check and a cycle budget. The engine knows nothing about KPIs — it walks whatever graph it is given.

In [3]:
# =============================================================================
# Cell 2  ·  Graph primitives
# =============================================================================
# Concept 7: a node graph makes the wiring EXPLICIT — which stages run, in what
# order, which run in parallel, where the decisions are, and where a failure
# sends control back.
#
# Lab 6's run plan was a list with dependency hints; the order was implicit in
# how the list was written. Here the edges are data, so the graph can be drawn,
# checked for unreachable nodes, and walked by an engine that knows nothing
# about this worklet.

import json, textwrap, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

from edss_core import DOMAINS, KPI_DEFINITIONS, REPORTING_PERIOD, ToolRouter
from edss_stack import (SKILL_REGISTRY, archive, retrieve_by_tags, server,
                        coordinator_client)

COORDINATOR, AGENT, JOIN, VALIDATION, GATE, PUBLISH = (
    "coordinator", "agent", "join", "validation", "gate", "publish")


class GraphError(Exception):
    pass


@dataclass
class Node:
    name: str
    kind: str
    label: str
    fn: Callable
    parallel_group: Optional[str] = None      # nodes sharing a group run together


@dataclass
class Edge:
    src: str
    dst: str
    condition: Optional[str] = None           # None = unconditional
    back_edge: bool = False                   # a loop back to an earlier node


class Graph:
    """An explicit node graph with parallel groups, conditional edges and back-edges."""

    def __init__(self, name: str, max_steps: int = 40):
        self.name = name
        self.nodes: Dict[str, Node] = {}
        self.edges: List[Edge] = []
        self.routers: Dict[str, Callable] = {}   # node -> picks the condition label
        self.max_steps = max_steps
        self.trace: List[Dict[str, Any]] = []
        self.state: Dict[str, Any] = {}

    # ---- construction ---------------------------------------------------
    def add_node(self, node: Node) -> "Graph":
        self.nodes[node.name] = node
        return self

    def add_edge(self, src: str, dst: str, condition: str = None,
                 back_edge: bool = False) -> "Graph":
        for n in (src, dst):
            if n not in self.nodes:
                raise GraphError(f"edge references unknown node '{n}'")
        self.edges.append(Edge(src, dst, condition, back_edge))
        return self

    def set_router(self, node: str, router: Callable) -> "Graph":
        """A router returns the CONDITION LABEL to follow out of this node."""
        self.routers[node] = router
        return self

    # ---- static checks --------------------------------------------------
    def successors(self, node: str, condition: str = None) -> List[str]:
        return [e.dst for e in self.edges
                if e.src == node and (e.condition is None or e.condition == condition)]

    def validate(self, entry: str) -> Dict[str, Any]:
        """Reachability and wiring problems, found before anything runs."""
        seen, stack = set(), [entry]
        while stack:
            n = stack.pop()
            if n in seen:
                continue
            seen.add(n)
            stack.extend(e.dst for e in self.edges if e.src == n)
        unreachable = sorted(set(self.nodes) - seen)
        dangling = sorted({e.dst for e in self.edges} - set(self.nodes))
        terminals = sorted(n for n in self.nodes
                           if not [e for e in self.edges if e.src == n])
        conditional_without_router = sorted(
            {e.src for e in self.edges if e.condition} - set(self.routers))
        return {"unreachable": unreachable, "dangling": dangling,
                "terminal_nodes": terminals,
                "conditional_edges_without_a_router": conditional_without_router,
                "ok": not (unreachable or dangling or conditional_without_router)}

    # ---- execution ------------------------------------------------------
    def _log(self, node: str, kind: str, note: str, ms: float, condition=None):
        self.trace.append({"step": len(self.trace) + 1, "node": node, "kind": kind,
                           "note": note, "condition_taken": condition,
                           "ms": round(ms, 2),
                           "at": datetime.now(timezone.utc).isoformat(timespec="seconds")})

    def run(self, entry: str, state: Dict[str, Any] = None) -> Dict[str, Any]:
        check = self.validate(entry)
        if not check["ok"]:
            raise GraphError(f"graph will not run: {check}")
        self.state = dict(state or {})
        current, steps = entry, 0

        while current is not None:
            steps += 1
            if steps > self.max_steps:
                raise GraphError(f"cycle budget of {self.max_steps} steps exhausted at "
                                 f"'{current}' — a back-edge is not converging")
            node = self.nodes[current]

            # --- parallel group: run every node in the group at once -------
            group = [n for n in self.nodes.values()
                     if node.parallel_group and n.parallel_group == node.parallel_group]
            t0 = time.perf_counter()
            if len(group) > 1:
                with ThreadPoolExecutor(max_workers=len(group)) as pool:
                    futures = {pool.submit(g.fn, self.state): g for g in group}
                    for fut, g in futures.items():
                        self.state[g.name] = fut.result()
                self._log(node.parallel_group, "parallel",
                          f"{len(group)} nodes in parallel: "
                          f"{[g.name for g in group]}",
                          (time.perf_counter() - t0) * 1000)
                current = self.successors(group[0].name)[0] if self.successors(group[0].name) else None
                continue

            # --- ordinary node ---------------------------------------------
            result = node.fn(self.state)
            self.state[node.name] = result
            condition = self.routers[node.name](self.state) if node.name in self.routers else None
            nxt = self.successors(node.name, condition)
            self._log(node.name, node.kind,
                      result.get("note", "") if isinstance(result, dict) else "",
                      (time.perf_counter() - t0) * 1000, condition)
            current = nxt[0] if nxt else None
        return self.state


print("Graph primitives ready: Node, Edge, parallel groups, conditional routers,")
print("back-edges, static reachability checks and a cycle budget.")


Graph primitives ready: Node, Edge, parallel groups, conditional routers,
back-edges, static reachability checks and a cycle budget.


### Cell 3 · The Worklet 15 graph

Chapter 15's node graph, wired as data. Note `n_report`: on the first pass it omits one source system. That injected defect is what makes the back-edge demonstrable rather than theoretical.

Note also the comment on `n_validation` — the engine stores each node's return value under the node's own name, so a node must **not** also write that key itself. Getting that wrong was a real `KeyError` while building this.

In [4]:
# =============================================================================
# Cell 3  ·  The Worklet 15 graph
# =============================================================================
# Chapter 15's node graph, wired as data: coordinator -> four domain agents in
# parallel -> KPI join -> report -> validation -> human gate -> publish, with a
# RED BACK-EDGE from validation to the report node.

router = ToolRouter(agent="graph")
REGEN = {"count": 0}          # how many times the back-edge has fired


def n_coordinator(state):
    reports = coordinator_client.read("department_reports", period=REPORTING_PERIOD)
    state["rows"] = reports["rows"]
    return {"note": f"{reports['row_count']} department rows, access confirmed"}


def _domain_node(domain):
    def run(state):
        review = SKILL_REGISTRY["review_domain"].run(router, domain=domain,
                                                     period=REPORTING_PERIOD)
        return {"domain": domain, "kpis": review["kpis"],
                "off_target": review["off_target"],
                "note": review["headline"]}
    return run


def n_kpi_join(state):
    kpis = [k for d in DOMAINS for k in state[f"{d.lower()}_agent"]["kpis"]]
    state["kpis"] = kpis
    return {"note": f"{len(kpis)} institution KPIs synthesised from 4 domain views"}


def n_report(state):
    # On the first pass the report omits the source system for one KPI — the
    # defect validation is there to catch. The back-edge repairs it.
    kpis = state["kpis"]
    rows = []
    for k in kpis:
        cited = k["source_system"]
        if REGEN["count"] == 0 and k["kpi_id"] == "KPI-R2":
            cited = ""                      # <- the injected defect
        rows.append({"kpi_id": k["kpi_id"], "name": k["name"], "value": k["value"],
                     "target": k["target"], "status": k["status"],
                     "source_system": cited,
                     "figure_status": k["figure_status"]})
    state["brief"] = {"rows": rows, "revision": REGEN["count"]}
    return {"note": f"brief drafted, revision {REGEN['count']}, {len(rows)} KPI rows"}


def n_validation(state):
    """The four Chapter 15 checks. Returns which KPIs failed, if any."""
    rows = state["brief"]["rows"]
    failing = [r["kpi_id"] for r in rows if not r["source_system"]]
    provisional_unlabelled = [r["kpi_id"] for r in rows
                              if r["figure_status"] != "final" and not r.get("caveat_ok", True)]
    # The engine stores a node's return value under the node's own name, so the
    # result IS state["validation"] — a node must not also write that key itself.
    return {"passed": not failing and not provisional_unlabelled,
            "failing": failing,
            "note": ("all checks passed" if not failing
                     else f"FAILED: {failing} cite no source system")}


def route_validation(state):
    return "pass" if state["validation"]["passed"] else "fail"


def n_regenerate(state):
    """Only the failing part is redone — not the whole run."""
    REGEN["count"] += 1
    return {"note": f"regenerating only {state['validation']['failing']} "
                    f"(attempt {REGEN['count']})"}


def n_human_gate(state):
    state["approval"] = {"by": "Dr. A. Principal", "decision": "approve"}
    return {"note": "Principal / Registrar approved for circulation"}


def route_gate(state):
    return state["approval"]["decision"]


def n_publish(state):
    state["published"] = {"version": 1, "to": "Governing Council"}
    return {"note": "versioned brief released to the Governing Council"}


g = Graph("worklet-15-executive-decision-support")
g.add_node(Node("coordinator", COORDINATOR, "apniLeap AURA Coordinator", n_coordinator))
for d in DOMAINS:
    g.add_node(Node(f"{d.lower()}_agent", AGENT, f"{d} Agent", _domain_node(d),
                    parallel_group="domain_agents"))
g.add_node(Node("kpi_agent", JOIN, "KPI Agent — synthesis", n_kpi_join))
g.add_node(Node("report_agent", AGENT, "Report Generation Agent", n_report))
g.add_node(Node("validation", VALIDATION, "Validation gate (hard)", n_validation))
g.add_node(Node("regenerate", AGENT, "Regenerate the failing KPI only", n_regenerate))
g.add_node(Node("human_gate", GATE, "Principal / Registrar review", n_human_gate))
g.add_node(Node("publish", PUBLISH, "Governed publish + audit", n_publish))

# The fan-out is expressed as four real edges. Leaving it implicit in the
# parallel group makes three agents unreachable — which the static check in
# validate() catches before a single node runs.
for d in DOMAINS:
    g.add_edge("coordinator", f"{d.lower()}_agent")
    g.add_edge(f"{d.lower()}_agent", "kpi_agent")
g.add_edge("kpi_agent", "report_agent")
g.add_edge("report_agent", "validation")
g.add_edge("validation", "human_gate", condition="pass")
g.add_edge("validation", "regenerate", condition="fail")
g.add_edge("regenerate", "report_agent", back_edge=True)     # <- the red back-edge
g.add_edge("human_gate", "publish", condition="approve")
g.add_edge("human_gate", "regenerate", condition="edit", back_edge=True)
g.set_router("validation", route_validation)
g.set_router("human_gate", route_gate)

print(f"Graph '{g.name}': {len(g.nodes)} nodes, {len(g.edges)} edges "
      f"({sum(1 for e in g.edges if e.back_edge)} back-edges, "
      f"{sum(1 for e in g.edges if e.condition)} conditional)")
print(f"\nStatic check before anything runs: {json.dumps(g.validate('coordinator'), indent=2)}")


Graph 'worklet-15-executive-decision-support': 11 nodes, 15 edges (2 back-edges, 4 conditional)

Static check before anything runs: {
  "unreachable": [],
  "dangling": [],
  "terminal_nodes": [
    "publish"
  ],
  "conditional_edges_without_a_router": [],
  "ok": true
}


### Cell 4 · Draw the graph

Because the edges are data, the graph renders itself — as ASCII and as Mermaid. The Mermaid block pastes straight into the review sheet or a README.

Legend follows Chapter 15: navy coordinator, light-blue agents, red validation, amber human gate, green governed publish, dashed back-edges.

In [5]:
# =============================================================================
# Cell 4  ·  Draw the graph
# =============================================================================
# Because the edges are data, the graph can be rendered — and a reviewer can
# check the wiring without reading the execution code.

LEGEND = {COORDINATOR: "navy   ", AGENT: "lt blue", JOIN: "lt blue",
          VALIDATION: "red    ", GATE: "amber  ", PUBLISH: "green  "}


def render_ascii(graph: Graph) -> str:
    out = [f"  {graph.name}", ""]
    groups = defaultdict(list)
    for n in graph.nodes.values():
        if n.parallel_group:
            groups[n.parallel_group].append(n.name)
    out.append("  [coordinator]  apniLeap AURA Coordinator")
    out.append("        |")
    out.append("        +--- fan-out, all four at once ---+")
    for name in groups["domain_agents"]:
        out.append(f"        |    [{name}]".ljust(46) + "|")
    out.append("        +---------------------------------+")
    out.append("        v")
    for name in ["kpi_agent", "report_agent", "validation"]:
        out.append(f"  [{name}]  {graph.nodes[name].label}")
        out.append("        |")
    out.append("        +-- fail --> [regenerate] --(back-edge)--> [report_agent]")
    out.append("        |")
    out.append("        +-- pass --> [human_gate]  Principal / Registrar")
    out.append("                          |")
    out.append("                          +-- edit --(back-edge)--> [regenerate]")
    out.append("                          |")
    out.append("                          +-- approve --> [publish]  governed, versioned")
    return "\n".join(out)


def render_mermaid(graph: Graph) -> str:
    lines = ["flowchart TD"]
    for n in graph.nodes.values():
        shape = ("{{%s}}" if n.kind == VALIDATION else
                 "[[%s]]" if n.kind == GATE else
                 "([%s])" if n.kind == PUBLISH else "[%s]")
        lines.append(f"    {n.name}{shape % n.label}")
    for e in graph.edges:
        label = f"|{e.condition}|" if e.condition else ""
        arrow = "-.->" if e.back_edge else "-->"
        lines.append(f"    {e.src} {arrow}{label} {e.dst}")
    return "\n".join(lines)


print(render_ascii(g))
print("\n" + "=" * 86)
print("The same graph as Mermaid — paste into any Markdown renderer:")
print("=" * 86)
print(render_mermaid(g))

print("\n" + "=" * 86)
print(f"{'node':<18}{'kind':<13}{'parallel group':<18}outgoing edges")
print("=" * 86)
for n in g.nodes.values():
    outs = [f"{e.dst}{'(' + e.condition + ')' if e.condition else ''}"
            f"{' [back]' if e.back_edge else ''}" for e in g.edges if e.src == n.name]
    print(f"{n.name:<18}{n.kind:<13}{n.parallel_group or '-':<18}{', '.join(outs) or '(terminal)'}")


  worklet-15-executive-decision-support

  [coordinator]  apniLeap AURA Coordinator
        |
        +--- fan-out, all four at once ---+
        |    [academic_agent]                 |
        |    [research_agent]                 |
        |    [placement_agent]                |
        |    [finance_agent]                  |
        +---------------------------------+
        v
  [kpi_agent]  KPI Agent — synthesis
        |
  [report_agent]  Report Generation Agent
        |
  [validation]  Validation gate (hard)
        |
        +-- fail --> [regenerate] --(back-edge)--> [report_agent]
        |
        +-- pass --> [human_gate]  Principal / Registrar
                          |
                          +-- edit --(back-edge)--> [regenerate]
                          |
                          +-- approve --> [publish]  governed, versioned

The same graph as Mermaid — paste into any Markdown renderer:
flowchart TD
    coordinator[apniLeap AURA Coordinator]
    academic_agent[Aca

### Cell 5 · Run it — validation fails, the back-edge fires

Ten steps. Watch steps 5–8: validation fails on `KPI-R2`, control routes to `regenerate`, back to `report_agent`, and validation passes on the second pass.

`domain_agents` and `kpi_agent` each appear **once** in the trace. Only the failing stage was redone.

In [6]:
# =============================================================================
# Cell 5  ·  Run the graph — validation fails, the back-edge fires
# =============================================================================
# The report node omits the source system for KPI-R2 on its first pass. That is
# exactly what Chapter 15's validation gate exists to catch ("every figure cites
# its source"), and the graph routes control back to fix only that.

print("=" * 86)
print("GRAPH RUN")
print("=" * 86)
final_state = g.run("coordinator")

print(f"  {'step':<6}{'node':<20}{'kind':<13}{'cond':<8}{'ms':>8}  note")
for e in g.trace:
    print(f"  {e['step']:<6}{e['node'][:18]:<20}{e['kind']:<13}"
          f"{(e['condition_taken'] or '-'):<8}{e['ms']:>8.2f}  {e['note'][:52]}")

visits = defaultdict(int)
for e in g.trace:
    visits[e["node"]] += 1
print(f"\n  back-edge fired         : {REGEN['count']} time(s)")
print(f"  nodes visited more than once: "
      f"{ {k: v for k, v in visits.items() if v > 1} }")
print(f"  KPI-R2 source after repair : "
      f"{[r['source_system'] for r in final_state['brief']['rows'] if r['kpi_id'] == 'KPI-R2'][0]!r}")
print(f"  validation final result    : "
      f"{'passed' if final_state['validation']['passed'] else 'failed'}")
print(f"  published                  : {final_state.get('published')}")

print("\n  What did NOT happen: the four domain agents did not re-run. The back-edge")
print("  returns to report_agent, so only the failing stage is redone — which is the")
print("  whole reason the loop is drawn as an edge instead of restarting the graph.")


GRAPH RUN
  step  node                kind         cond          ms  note
  1     coordinator         coordinator  -           1.49  8 department rows, access confirmed
  2     domain_agents       parallel     -           1.12  4 nodes in parallel: ['academic_agent', 'research_ag
  3     kpi_agent           join         -           0.01  9 institution KPIs synthesised from 4 domain views
  4     report_agent        agent        -           0.01  brief drafted, revision 0, 9 KPI rows
  5     validation          validation   fail        0.00  FAILED: ['KPI-R2'] cite no source system
  6     regenerate          agent        -           0.00  regenerating only ['KPI-R2'] (attempt 1)
  7     report_agent        agent        -           0.00  brief drafted, revision 1, 9 KPI rows
  8     validation          validation   pass        0.00  all checks passed
  9     human_gate          gate         approve     0.00  Principal / Registrar approved for circulation
  10    publish             publ

### Cell 6 · What the graph refuses

A non-converging back-edge (stopped by the cycle budget), an unreachable node, an edge to a node that does not exist, and the structural guarantee that **publish is reachable only through the human gate** — there is no path from validation or any agent straight to publish.

In [7]:
# =============================================================================
# Cell 6  ·  Guardrails — what the graph refuses
# =============================================================================

print("=" * 86)
print("1 · A BACK-EDGE THAT NEVER CONVERGES IS STOPPED")
print("=" * 86)
stuck = Graph("never-converges", max_steps=12)
stuck.add_node(Node("draft", AGENT, "draft", lambda s: {"note": "drafted"}))
stuck.add_node(Node("check", VALIDATION, "always fails",
                    lambda s: {"passed": False, "note": "still failing"}))
stuck.add_node(Node("fix", AGENT, "fix", lambda s: {"note": "fixing"}))
stuck.add_node(Node("done", PUBLISH, "done", lambda s: {"note": "done"}))
stuck.add_edge("draft", "check")
stuck.add_edge("check", "done", condition="pass")
stuck.add_edge("check", "fix", condition="fail")
stuck.add_edge("fix", "draft", back_edge=True)
stuck.set_router("check", lambda s: "pass" if s["check"]["passed"] else "fail")
try:
    stuck.run("draft")
    print("  [!!] a non-converging loop ran forever")
except GraphError as exc:
    print(f"  [REFUSED] {exc}")
    print(f"            steps taken before the stop: {len(stuck.trace)}")

print("\n" + "=" * 86)
print("2 · AN UNREACHABLE NODE IS FOUND BEFORE THE GRAPH RUNS")
print("=" * 86)
orphan = Graph("has-an-orphan")
orphan.add_node(Node("start", COORDINATOR, "start", lambda s: {"note": "go"}))
orphan.add_node(Node("finish", PUBLISH, "finish", lambda s: {"note": "end"}))
orphan.add_node(Node("forgotten_agent", AGENT, "never wired in",
                     lambda s: {"note": "never runs"}))
orphan.add_edge("start", "finish")
report = orphan.validate("start")
print(f"  static check: {json.dumps(report)}")
try:
    orphan.run("start")
    print("  [!!] the graph ran with an unreachable node")
except GraphError as exc:
    print(f"  [REFUSED] {str(exc)[:110]}…")
print("\n  This is the same class of bug the real graph had while it was being wired:")
print("  the fan-out to three domain agents was implicit in the parallel group, and")
print("  validate() reported them unreachable until four real edges were added.")

print("\n" + "=" * 86)
print("3 · AN EDGE TO A NODE THAT DOES NOT EXIST IS REFUSED AT BUILD TIME")
print("=" * 86)
try:
    g.add_edge("publish", "governing_council_dinner")
    print("  [!!] an edge to an undefined node was accepted")
except GraphError as exc:
    print(f"  [REFUSED] {exc}")

print("\n" + "=" * 86)
print("4 · PUBLISH IS ONLY REACHABLE THROUGH THE HUMAN GATE")
print("=" * 86)
into_publish = [e for e in g.edges if e.dst == "publish"]
print(f"  edges into publish: {[(e.src, e.condition) for e in into_publish]}")
print(f"  every one of them comes from the human gate: "
      f"{all(e.src == 'human_gate' for e in into_publish)}")
print("  There is no path from validation, or from any agent, straight to publish.")


1 · A BACK-EDGE THAT NEVER CONVERGES IS STOPPED
  [REFUSED] cycle budget of 12 steps exhausted at 'draft' — a back-edge is not converging
            steps taken before the stop: 12

2 · AN UNREACHABLE NODE IS FOUND BEFORE THE GRAPH RUNS
  static check: {"unreachable": ["forgotten_agent"], "dangling": [], "terminal_nodes": ["finish", "forgotten_agent"], "conditional_edges_without_a_router": [], "ok": false}
  [REFUSED] graph will not run: {'unreachable': ['forgotten_agent'], 'dangling': [], 'terminal_nodes': ['finish', 'forgott…

  This is the same class of bug the real graph had while it was being wired:
  the fan-out to three domain agents was implicit in the parallel group, and
  validate() reported them unreachable until four real edges were added.

3 · AN EDGE TO A NODE THAT DOES NOT EXIST IS REFUSED AT BUILD TIME
  [REFUSED] edge references unknown node 'governing_council_dinner'

4 · PUBLISH IS ONLY REACHABLE THROUGH THE HUMAN GATE
  edges into publish: [('human_gate', 'approve'

### Cell 7 · Acceptance checks — the rubric, executed

In [8]:
# =============================================================================
# Cell 7  ·  Acceptance checks — the rubric, executed
# =============================================================================
# Worklet 15, Lab 7: wire the specialist agents into an explicit node graph —
# stages, parallel branches, human gates and a validation back-edge.

checks = []


def check(name, condition, evidence=""):
    checks.append((name, bool(condition), evidence))


check("G1  the wiring is data, not control flow",
      len(g.edges) == 15 and all(isinstance(e, Edge) for e in g.edges),
      f"{len(g.nodes)} nodes and {len(g.edges)} edges, inspectable and renderable")

check("G2  the graph is statically checked before it runs",
      g.validate("coordinator")["ok"] and report["unreachable"] == ["forgotten_agent"],
      "the real graph passes; the orphan graph reports forgotten_agent unreachable")

check("G3  an edge to an undefined node is refused at build time",
      "governing_council_dinner" not in {e.dst for e in g.edges},
      "add_edge raised GraphError rather than storing a dangling edge")

check("G4  the four domain agents ran in parallel, as one step",
      any(e["kind"] == "parallel" and "4 nodes" in e["note"] for e in g.trace),
      next(e["note"] for e in g.trace if e["kind"] == "parallel")[:60])

check("G5  the join waits for all four domain views",
      len(final_state["kpis"]) == len(KPI_DEFINITIONS),
      f"kpi_agent synthesised {len(final_state['kpis'])} KPIs from 4 domain agents")

check("G6  validation is a real gate — it failed the first pass",
      any(e["node"] == "validation" and e["condition_taken"] == "fail" for e in g.trace),
      next(e["note"] for e in g.trace if e["node"] == "validation"))

check("G7  the back-edge fired and control returned to the report node",
      REGEN["count"] == 1
      and [e["node"] for e in g.trace].count("report_agent") == 2,
      f"regenerate -> report_agent taken {REGEN['count']} time(s)")

check("G8  ONLY the failing stage was redone",
      [e["node"] for e in g.trace].count("domain_agents") == 1
      and [e["node"] for e in g.trace].count("kpi_agent") == 1,
      "domain agents and the KPI join each ran exactly once across both passes")

check("G9  the defect validation existed to catch was actually repaired",
      all(r["source_system"] for r in final_state["brief"]["rows"]),
      "every KPI row in the final brief cites a source system")

check("G10 the second pass then passed the gate",
      final_state["validation"]["passed"],
      "validation -> pass -> human_gate")

check("G11 publish is reachable only through the human gate",
      all(e.src == "human_gate" for e in g.edges if e.dst == "publish"),
      f"edges into publish: {[(e.src, e.condition) for e in g.edges if e.dst == 'publish']}")

check("G12 the human gate has a route for edit as well as approve",
      {e.condition for e in g.edges if e.src == "human_gate"} == {"approve", "edit"},
      "approve -> publish; edit -> regenerate (a second back-edge)")

check("G13 a non-converging back-edge is stopped by the cycle budget",
      len(stuck.trace) <= stuck.max_steps,
      f"the always-failing graph was halted after {len(stuck.trace)} steps")

check("G14 the run is traceable end to end",
      len(g.trace) == 10 and g.trace[0]["node"] == "coordinator"
      and g.trace[-1]["node"] == "publish",
      f"{len(g.trace)} steps: coordinator -> … -> publish")

print("=" * 86)
print("LAB 7 ACCEPTANCE CHECKS")
print("=" * 86)
for name, ok, ev in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")
    if ev:
        print(f"          {ev}")
failed = [n for n, ok, _ in checks if not ok]
print("=" * 86)
print(f"  {len(checks) - len(failed)}/{len(checks)} checks passed.")
assert not failed, f"FAILED: {failed}"
print("  Lab 7 acceptance criteria satisfied.")


LAB 7 ACCEPTANCE CHECKS
  [PASS]  G1  the wiring is data, not control flow
          11 nodes and 15 edges, inspectable and renderable
  [PASS]  G2  the graph is statically checked before it runs
          the real graph passes; the orphan graph reports forgotten_agent unreachable
  [PASS]  G3  an edge to an undefined node is refused at build time
          add_edge raised GraphError rather than storing a dangling edge
  [PASS]  G4  the four domain agents ran in parallel, as one step
          4 nodes in parallel: ['academic_agent', 'research_agent', 'p
  [PASS]  G5  the join waits for all four domain views
          kpi_agent synthesised 9 KPIs from 4 domain agents
  [PASS]  G6  validation is a real gate — it failed the first pass
          FAILED: ['KPI-R2'] cite no source system
  [PASS]  G7  the back-edge fired and control returned to the report node
          regenerate -> report_agent taken 1 time(s)
  [PASS]  G8  ONLY the failing stage was redone
          domain agents and the 

### Cell 8 · Deliverables

In [9]:
# =============================================================================
# Cell 8  ·  Deliverables
# =============================================================================
outdir = Path("lab7_artifacts")
outdir.mkdir(exist_ok=True)

artifacts = {
    "lab7_graph_definition.json": {
        "name": g.name,
        "nodes": [{"name": n.name, "kind": n.kind, "label": n.label,
                   "parallel_group": n.parallel_group} for n in g.nodes.values()],
        "edges": [{"src": e.src, "dst": e.dst, "condition": e.condition,
                   "back_edge": e.back_edge} for e in g.edges],
        "static_check": g.validate("coordinator")},
    "lab7_graph.mmd": render_mermaid(g),
    "lab7_graph.txt": render_ascii(g),
    "lab7_execution_trace.json": g.trace,
    "lab7_final_brief_rows.json": final_state["brief"],
    "lab7_backedge_evidence.json": {
        "regenerations": REGEN["count"],
        "node_visits": dict(visits),
        "validation_first_pass": next(e["note"] for e in g.trace
                                      if e["node"] == "validation"),
        "validation_final": final_state["validation"],
        "domain_agents_reran": [e["node"] for e in g.trace].count("domain_agents") > 1},
    "lab7_acceptance_checks.json": [{"check": n, "passed": ok, "evidence": e}
                                    for n, ok, e in checks],
}
for fname, obj in artifacts.items():
    (outdir / fname).write_text(obj if isinstance(obj, str)
                                else json.dumps(obj, indent=2, default=str))

print("=" * 86)
print("ARTIFACTS WRITTEN")
print("=" * 86)
for fname in artifacts:
    print(f"  {outdir / fname}  ({(outdir / fname).stat().st_size:,} bytes)")
print(f"\n  Nodes / edges     : {len(g.nodes)} / {len(g.edges)} "
      f"({sum(1 for e in g.edges if e.back_edge)} back-edges)")
print(f"  Steps executed    : {len(g.trace)}")
print(f"  Back-edge fired   : {REGEN['count']}")
print(f"  Acceptance checks : {sum(ok for _, ok, _ in checks)}/{len(checks)} passed")


ARTIFACTS WRITTEN
  lab7_artifacts/lab7_graph_definition.json  (3,598 bytes)
  lab7_artifacts/lab7_graph.mmd  (965 bytes)
  lab7_artifacts/lab7_graph.txt  (860 bytes)
  lab7_artifacts/lab7_execution_trace.json  (2,173 bytes)
  lab7_artifacts/lab7_final_brief_rows.json  (2,128 bytes)
  lab7_artifacts/lab7_backedge_evidence.json  (420 bytes)
  lab7_artifacts/lab7_acceptance_checks.json  (2,361 bytes)

  Nodes / edges     : 11 / 15 (2 back-edges)
  Steps executed    : 10
  Back-edge fired   : 1
  Acceptance checks : 14/14 passed


---

## Where Lab 7 leaves the worklet

**Nodes activated:** the full Chapter 15 graph — coordinator, four domain agents, KPI join, report generation, validation, human gate, governed publish.

**What exists now:** a pipeline whose wiring is inspectable, checkable and drawable, with the failure path wired in rather than handled by an exception somewhere.

**What Lab 8 adds:** the swarm. The four domain agents already run in one parallel step. Lab 8 makes that parallelism *fault-tolerant and reconcilable* — what happens when one agent fails, when two agents disagree about the same figure, and how partial results are merged into one institution view rather than abandoned.

> Lab 8 acceptance (Chapter 15): parallelise the per-unit work and **merge the best results**, resolving conflicts.